# 00 · Setup & load data from Hugging Face

**Welcome!** This is the start of the onboarding notebook series for the
*Persona-Conditional Steering Vectors* project. Read
[`docs/overview.md`](../docs/overview.md) first for the research question and findings.

**You do not need a GPU.** These notebooks load the *pre-computed* steering vectors
(~110 MB) from the published dataset and reproduce the headline analyses on CPU.

What the series covers:
| Notebook | Topic |
|---|---|
| **00** | Setup, download vectors from HF, load into memory *(this one)* |
| 01 | Explore the personas and traits |
| 02 | Vector geometry: cross-persona transfer matrices |
| 03 | Shared vs persona-specific variance decomposition |
| 04 | Reproduce the key figures |

## 1. Install the package

From the repo root, in a Python **3.10+** environment:

```bash
pip install -e .
```

The cell below installs the few extras these notebooks need (safe to re-run).

In [ ]:
# One-time: install the project (editable) + notebook extras.
# Comment out once your environment is set up.
# %pip install -e .. huggingface_hub matplotlib seaborn scikit-learn

## 2. Download the pre-computed vectors

The canonical data lives on the Hub:
[`girishgupta/persona-steering-activations`](https://huggingface.co/datasets/girishgupta/persona-steering-activations).
We pull only the small, ready-to-analyse pieces: the IV + CAA **steering vectors**,
the **persona** YAMLs, and the **trait prompts**. (The full activations are ~65 GB —
you don't need them for analysis.)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

DATA = Path('hf_data')  # local cache for this notebook series
snapshot_download(
    repo_id='girishgupta/persona-steering-activations',
    repo_type='dataset',
    local_dir=str(DATA),
    allow_patterns=['vectors/*', 'caa_vectors/*', 'personas/*', 'prompts/*'],
)
print('Downloaded into', DATA.resolve())
print('IV vectors :', len(list((DATA / 'vectors').glob('*.pt'))), 'files')
print('CAA vectors:', len(list((DATA / 'caa_vectors').glob('*.pt'))), 'files')

## 3. Load vectors into memory

Each `.pt` is one persona×trait steering vector with shape `(46, 4608)` —
one row per transformer layer. We work at **layer 22** (the analysis layer used
throughout the project). The helper below is robust to both on-disk formats.

In [ ]:
import torch
from pathlib import Path
from persona_steering.utils import parse_persona_trait_from_stem

def load_vector_dir(vectors_dir, layer=22):
    """Load all steering vectors from a dir of .pt files at one layer.

    Robust to two on-disk formats:
      * dict payload  {'vector': Tensor[L, D], 'persona':..., 'trait':...}
      * a raw Tensor[L, D]
    Returns {(persona, trait): Tensor[D]} for the requested layer.
    """
    vectors_dir = Path(vectors_dir)
    out = {}
    for pt in sorted(vectors_dir.glob('*.pt')):
        obj = torch.load(pt, map_location='cpu', weights_only=False)
        vec = obj['vector'] if isinstance(obj, dict) and 'vector' in obj else obj
        vec = vec.float()
        persona, trait = parse_persona_trait_from_stem(pt.stem)
        if persona and trait and layer < vec.shape[0]:
            out[(persona, trait)] = vec[layer]
    return out

In [ ]:
LAYER = 22
iv = load_vector_dir(DATA / 'vectors', layer=LAYER)
caa = load_vector_dir(DATA / 'caa_vectors', layer=LAYER)

personas = sorted({p for (p, t) in iv})
traits   = sorted({t for (p, t) in iv})
print(f'{len(iv)} IV vectors | {len(personas)} personas x {len(traits)} traits')
print('personas:', personas)
print('traits  :', traits)
example = iv[(personas[0], traits[0])]
print('one vector shape (single layer):', tuple(example.shape))

You now have `iv` and `caa` dictionaries mapping `(persona, trait) -> Tensor[4608]`.
The next notebooks reuse this exact loading pattern. ➡️ Continue to
**[01 · Personas & traits](01_personas_and_traits.ipynb)**.